# 11_00 — Export the offline Kaggle inference bundle

**CLOSED-WORLD / CLASS-1 ONLY.** Candidates are the training structure library. There is no
external candidate expansion and no Mode B. Submission 1 is an anchor, not the final solution.

This notebook writes `bundle/`: structures, adducts, ppm windows, reference metadata, CSR index,
peak store, models, config, code, README and a sha256 manifest. It can run **before** the
scaling track finishes. It uses the v5 frozen spectrum model if
`outputs/v5/freeze/spectrum_model.json` exists, and otherwise the historical v4b V1. Swapping the
model later only re-runs section 6 and the manifest refresh.

The reference-library export streams all ~2.5M train spectra once (peak cleaning, deterministic
top-100 and T1 peak hash). That is the only slow step, and it is skipped when an export with
the same inputs already exists.

In [3]:
import sys, json, hashlib
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
for _mod in [m for m in list(sys.modules) if m.split('.')[0] in ('casmi', 'casmi_infer')]:
    del sys.modules[_mod]                 # a long-lived kernel must not export with stale module objects

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.io.artifacts import load_artifact
from casmi.candidates.generator import CandidateGenerationConfig
from casmi.chemistry.adducts import neutral_mass_from_precursor
from casmi.qcr.context import v4b_paths, v5_paths
from casmi.qcr.fingerprint import code_version
from casmi.qcr.settlement import load_settlement_or_refuse
from casmi.ranking.lambdamart import BASE_FEATURES
from casmi import bundle_export as bx
from casmi_infer.adducts import AdductRules
from casmi_infer.validation import check_bundle_invariants, sha256_file, verify_bundle

paths = get_project_paths()
vp, D = v4b_paths(paths), v5_paths(paths)
BUNDLE = D['bundle']
BUNDLE.mkdir(parents=True, exist_ok=True)
REBUILD_REFERENCE_LIBRARY = False     # True forces the ~2.5M-spectrum peak-store re-export
COMPUTE_PEAK_HASH = True              # T1 exact-mirror check at inference needs it (slowest part of the export)
PROMOTE_LOCKED_AGGREGATOR = True      # v6.3: ship the MOL_DEV-locked A7 (+ frozen calibration). False = explicit RRF baseline bundle
settlement = load_settlement_or_refuse(vp.settlement_json)
SIM_CFG = settlement['similarity_config']

# ---- the FROZEN spectrum model only (PROVISIONAL / historical models are refused) -------------------------
from casmi.ranking.freeze import require_frozen_spectrum_model
freeze_path = D['freeze'] / 'spectrum_model.json'
freeze = require_frozen_spectrum_model(freeze_path)
MODEL_DIR, MODEL_ID, FEATURE_NAMES = Path(freeze['model_dir']), freeze['model_id'], freeze['feature_names']
MODEL_EXTRA = {'freeze_status': freeze.get('freeze_status'), 'spectrum_model_json_sha256': sha256_file(freeze_path),
               'model_freeze_status': freeze.get('model_freeze_status', freeze.get('freeze_status')),
               'validation_realism_status': freeze.get('validation_realism_status'),
               'training_protocol': freeze.get('protocol'), 'training_protocol_hash': freeze.get('protocol_hash'),
               'inference_eligibility': 'production: exclude_sources={}, T1 by peak hash, T2 not applied (deployment gap WAIVED_PENDING_EVIDENCE)'}
assert list(FEATURE_NAMES) == list(BASE_FEATURES), 'frozen feature order differs from the V1 recipe'

# ---- v6.3: the LOCKED molecule aggregator (A7) + its frozen calibration; any inconsistency STOPS (no RRF fallback) ----
AGG_OUT = paths.outputs / 'v6' / 'aggregation'
if PROMOTE_LOCKED_AGGREGATOR:
    PROMOTION = bx.load_locked_aggregator(AGG_OUT / 'selected_aggregator.json', AGG_OUT / 'aggregator_lock_v6_2.json',
                                          AGG_OUT / 'calibration.json', freeze)
    AGGREGATOR_CFG, BUNDLE_VERSION = PROMOTION['aggregator'], bx.BUNDLE_VERSION
    print('promoting locked aggregator:', AGGREGATOR_CFG['name'], '| T =', PROMOTION['temperature'], '| artifacts:', PROMOTION['sha256'])
else:
    PROMOTION, AGGREGATOR_CFG, BUNDLE_VERSION = None, dict(bx.RRF_BASELINE_AGGREGATOR), 'v2-RRF-baseline'
    print('EXPLICIT RRF baseline bundle (PROMOTE_LOCKED_AGGREGATOR = False)')

# ---- never overwrite a previous bundle version silently: archive its identity files first -------------------
ARCHIVED = bx.archive_bundle_metadata(BUNDLE, D['root'] / 'bundle_archive')
print('previous bundle archived to:', ARCHIVED)
print('model to export:', MODEL_ID, '<-', MODEL_DIR, '| bundle version:', BUNDLE_VERSION)

promoting locked aggregator: MOST_CONFIDENT_SPECTRUM | T = 1.1947045372735254 | artifacts: {'selected_aggregator.json': '78b03cc2d8267d0a7fbdbb50a06cbb4e3cfa90b870b34bb236fce66465b6dd40', 'calibration.json': '9bc8f9703fdcb6633813f2b3eb098db362db48c6d04ebda9c8b05be1dea61319', 'aggregator_lock.json': 'c97cd00c60ce2458e01473b74d7260865d132a55483f340b4a39cc6b7e23ec0f'}
previous bundle archived to: C:\Users\myben\OneDrive\Documents\Enveda\outputs\v5\bundle_archive\128372cae78ad88b-V1_TL_1K__128372cae78ad88b
model to export: V1_TL_1K_TESTSIM_STRICT <- C:\Users\myben\OneDrive\Documents\Enveda\outputs\v5\models\V1_TL_1K_TESTSIM_STRICT | bundle version: v2-A7


## 1. Structures: the exact training candidate library (mass variants, sorted by float64 mass)

In [4]:
mmv = pd.read_parquet(paths.interim / 'candidate_generation' / 'molecule_mass_variants.parquet')
structures, conn = bx.export_structures(mmv, BUNDLE)
print(f'{len(structures):,} mass-variant rows, {len(conn):,} connectivities')

274,330 mass-variant rows, 274,195 connectivities


## 2. Adducts (serialized from the training parser) + parity against it

In [5]:
adduct_ref = pd.read_parquet(paths.processed / 'adduct_reference.parquet')
test_meta = pd.read_parquet(paths.processed / 'test_spectrum_metadata.parquet')
adducts_payload = bx.export_adducts(adduct_ref['adduct'].astype(str), test_meta['adduct'].astype(str).unique(), BUNDLE)
rules = AdductRules.load(BUNDLE / 'adducts.json')
mz_grid = np.linspace(80.0, 2000.0, 97)
parity_bad = [(a, mz) for a in rules.rules for mz in mz_grid
              if (rules.neutral_mass(mz, a) is None) != (neutral_mass_from_precursor(mz, a) is None)
              or (rules.neutral_mass(mz, a) is not None and rules.neutral_mass(mz, a) != neutral_mass_from_precursor(mz, a))]
print(f'adduct rules: {len(rules.rules)} | unsupported observed test adducts: {adducts_payload["unsupported_observed_test_adducts"]}')
print('adduct export parity violations (exact equality required):', len(parity_bad))
assert not parity_bad

adduct rules: 121 | unsupported observed test adducts: []
adduct export parity violations (exact equality required): 0


## 3. Retrieval windows (from the training contract)

In [6]:
with open(paths.processed / 'candidate_generation' / 'candidate_generation_contract.json') as f:
    contract = json.load(f)
ppm_payload = bx.export_ppm_windows(contract, adducts_payload, BUNDLE, CandidateGenerationConfig())
print({k: v for k, v in ppm_payload.items() if k != 'per_adduct_primary_ppm'})

{'source': 'candidate_generation_contract.json molecule_policy.tolerance_ppm (global; the adaptive per-group policy was evaluated in notebook 03 but not adopted by the contract)', 'default_primary_ppm': 50.0, 'max_standard_ppm': 50.0, 'fallback_ppm': [100.0, 200.0], 'nearest_n': 25, 'fallback_sequence': ['primary', 'ppm_100', 'ppm_200', 'nearest_25'], 'fallback_note': 'fallback levels are INFERENCE-ONLY (training never needed them: every training query had a non-empty primary window by construction); every use is logged'}


## 4. Reference library: metadata, CSR index, similarity-representation peak store

In [7]:
ref_files = ['ref_meta.parquet', 'ref_index_offsets.npy', 'ref_index_ids.npy', 'ref_peaks_offsets.npy', 'ref_peaks_mz.npy', 'ref_peaks_int.npy']
train_meta = load_artifact('train_spectrum_metadata')
lib_inputs = {'train.parquet': f'{paths.train.stat().st_size}|{paths.train.stat().st_mtime:.0f}', 'max_peaks': SIM_CFG['max_peaks_similarity'],
              'n_train_rows': len(train_meta), 'peak_hash': COMPUTE_PEAK_HASH, 'connectivity_table_sha256': sha256_file(BUNDLE / 'connectivities.parquet')}
stamp = BUNDLE / '_reference_library_inputs.json'
reuse = (not REBUILD_REFERENCE_LIBRARY and stamp.exists() and all((BUNDLE / f).exists() for f in ref_files)
         and json.loads(stamp.read_text(encoding='utf-8')) == lib_inputs)
if reuse:
    print('reference library: reusing the existing export (inputs unchanged)')
else:
    info = bx.export_reference_library(train_meta, paths.train, conn, BUNDLE, max_peaks=SIM_CFG['max_peaks_similarity'], compute_hash=COMPUTE_PEAK_HASH)
    stamp.write_text(json.dumps(lib_inputs, indent=2), encoding='utf-8')
    print(info)
problems = check_bundle_invariants(structures, conn, np.load(BUNDLE / 'ref_index_offsets.npy'), np.load(BUNDLE / 'ref_peaks_offsets.npy'), len(train_meta))
print('bundle invariants:', problems or 'OK')
assert not problems

reference library: reusing the existing export (inputs unchanged)
bundle invariants: OK


## 5. Peak-store spot check: exported arrays == training similarity representation

In [8]:
from casmi.io.parquet import load_rows_by_offset
from casmi.spectra.preprocessing import remove_invalid_peaks, truncate_top_peaks
from casmi_infer.reference_index import ReferenceLibrary
lib = ReferenceLibrary(BUNDLE)
rng = np.random.default_rng(42)
sample_rows = np.sort(rng.choice(len(train_meta), size=500, replace=False))
raw = load_rows_by_offset(paths.train, ['ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz'], sample_rows).rename(columns={'_row_offset': 'row_offset'})
bad = 0
for r in raw.itertuples(index=False):
    mz, it = remove_invalid_peaks(r.ms2_mzs, r.ms2_normalized_intensities)
    s_mz, s_it = truncate_top_peaks(mz, it, max_peaks=SIM_CFG['max_peaks_similarity'])
    p = lib.peaks(int(r.row_offset))
    bad += int(not (np.array_equal(p['mzs'], s_mz) and np.array_equal(p['intensities'], s_it) and p['precursor_mz'] == float(r.precursor_mz)))
print('peak-store spot-check mismatches (500 random references):', bad)
assert bad == 0

peak-store spot-check mismatches (500 random references): 0


## 6. Models (swap point) + config + code + README + manifest

In [9]:
from casmi.ranking.freeze import feature_order_hash
model_info = bx.export_models(MODEL_DIR, BUNDLE, MODEL_ID, FEATURE_NAMES, tag='v1', extra_info=MODEL_EXTRA)
MODEL_CHECK = bx.verify_bundle_models_match_freeze(BUNDLE, freeze)          # byte-identical to the freeze record, or STOP
print('models match the freeze record:', MODEL_CHECK)
CALIBRATION = bx.export_aggregator_artifacts(BUNDLE, PROMOTION) if PROMOTION else None
SMILES_VALIDATION = bx.validate_representative_smiles(conn, out_path=BUNDLE / 'smiles_validation.json')
print('representative SMILES (RDKit, export time):', {k: v for k, v in SMILES_VALIDATION.items() if k != 'failed_smiles'})
if SMILES_VALIDATION['n_failed']:
    print('WARNING: these representative SMILES do not parse; the Kaggle validator refuses to emit them:', SMILES_VALIDATION['failed_smiles'][:20])
CANDIDATE_UNIVERSE = bx.candidate_universe_version(BUNDLE)
config = bx.build_config(SIM_CFG, aggregator=AGGREGATOR_CFG, bundle_version=BUNDLE_VERSION,
                         model={'model_id': MODEL_ID, 'feature_order': list(FEATURE_NAMES), 'feature_order_hash': feature_order_hash(FEATURE_NAMES),
                                'protocol': freeze['protocol'], 'protocol_hash': freeze['protocol_hash']},
                         calibration=CALIBRATION, candidate_universe_version=CANDIDATE_UNIVERSE)     # CONFIG_HASH computed HERE
(BUNDLE / 'config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
code_files = bx.export_code(REPO_ROOT / 'src', BUNDLE)
(BUNDLE / 'README.md').write_text(bx.README, encoding='utf-8')
source_artifacts = {
    'molecule_mass_variants.parquet': sha256_file(paths.interim / 'candidate_generation' / 'molecule_mass_variants.parquet'),
    'adduct_reference.parquet': sha256_file(paths.processed / 'adduct_reference.parquet'),
    'candidate_generation_contract.json': sha256_file(paths.processed / 'candidate_generation' / 'candidate_generation_contract.json'),
    'train_spectrum_metadata.parquet': sha256_file(paths.processed / 'train_spectrum_metadata.parquet'),
    'train.parquet (size|mtime)': lib_inputs['train.parquet'],
    'cache_settlement.evidence_fingerprint': settlement['evidence_fingerprint'],
    'model_files': model_info['file_sha256'],
    'aggregator_promotion': PROMOTION['sha256'] if PROMOTION else None,
}
BUNDLE_VERSION_FULL = f"{BUNDLE_VERSION}-{config['CONFIG_HASH']}-{MODEL_ID}"
manifest = bx.refresh_manifest(BUNDLE, source_artifacts=source_artifacts, code_version=code_version(),
                               extra={'model_id': MODEL_ID, 'bundle_version': BUNDLE_VERSION_FULL, 'previous_version_archive': str(ARCHIVED) if ARCHIVED else None})
print('CONFIG_HASH', config['CONFIG_HASH'], '| bundle_version', BUNDLE_VERSION_FULL, '| aggregator', config['aggregator_name'],
      '| candidate universe', CANDIDATE_UNIVERSE, '| files:', len(manifest['files']), '| code files:', len(code_files))

models match the freeze record: {'model_id': 'V1_TL_1K_TESTSIM_STRICT', 'n_folds': 5, 'feature_order_hash': '21322da841f64194'}
representative SMILES (RDKit, export time): {'n_checked': 274195, 'n_failed': 0, 'rdkit_version': '2026.03.6'}
CONFIG_HASH 60174e39a2a3c6b4 | bundle_version v2-A7-60174e39a2a3c6b4-V1_TL_1K_TESTSIM_STRICT | aggregator MOST_CONFIDENT_SPECTRUM | candidate universe closed_world_train_library:f69db15509fd:8057af30f1f2 | files: 55 | code files: 28


## 7. Verify (sha256 of every file + CONFIG_HASH) and size report

In [10]:
_, _, problems = verify_bundle(BUNDLE)
print('bundle verification:', problems or 'OK')
assert not problems
sizes = pd.DataFrame([{'file': k, 'MB': (BUNDLE / k).stat().st_size / 1e6} for k in manifest['files']]).sort_values('MB', ascending=False)
display(sizes.head(15)); print(f"total {sizes['MB'].sum() / 1e3:.2f} GB")
print('Upload bundle/ MANUALLY as a private Kaggle dataset; nothing here uploads anything.')

bundle verification: OK


,file,MB
50,ref_peaks_mz.npy,1060.644624
49,ref_peaks_int.npy,1060.644624
0,_ref_peaks_int.f8,576.217120
1,_ref_peaks_mz.f8,576.217120
48,ref_meta.parquet,224.623066
51,ref_peaks_offsets.npy,20.317000
46,ref_index_ids.npy,20.316992
53,structures.parquet,17.276176
36,connectivities.parquet,12.446093
54,structures_mass.npy,2.194768


total 3.58 GB
Upload bundle/ MANUALLY as a private Kaggle dataset; nothing here uploads anything.


## 8. Bundle self-test fixture (`bundle/selftest/`, format 2), consumed by the Kaggle notebook before any hidden-test inference

About 50 HOST queries are chosen deterministically: stratified over adduct, pool-size quartile,
truth-reference bin and collision-energy presence, plus 10 large-pool multi-reference queries.
Everything is computed with the **numpy** backend, the path validated against training by `11_01` P1/P2:

* **Spectrum level:** candidates, features, every V1 fold score, the fold-mean score and ranks.
* **Molecule level (v6.3):** the fixture spectra are grouped into deterministic pseudo-molecules (sizes 3, 1,
  2, 4 cycling). The fixture then freezes the calibrated probabilities (frozen T), the A7 selected spectrum
  and its confidence, and the final Top-25 connectivity order of the *configured* aggregator.

The fixture reuses the exported reference library, so no peaks are duplicated, and Kaggle needs nothing
outside the bundle to run it. The expected features are also cross-checked here against the frozen HOST
training features. The self-test refuses a fixture that is stale in any of these ways: a different
CONFIG_HASH, different model files, a different feature order, a different temperature, or format 1.

In [11]:
from casmi_infer.pipeline import Bundle
from casmi_infer.selftest import build_fixture, select_fixture_queries
from casmi_infer.features import SPECTRAL_FEATURES
host_q = pd.read_parquet(vp.host_processed / 'host_holdout_spectra.parquet')
host_feats = pd.read_parquet(settlement['evidence_artifacts']['host_features_mirror_aware'])
pool_size = host_feats.groupby('query_id').size()
truth_refs = host_feats[host_feats['is_true_candidate']].set_index('query_id')['n_reference_spectra']
fx_ids = select_fixture_queries(host_q[['query_id', 'adduct']].assign(ce_known=host_q['ce_mean'].notna()), pool_size, truth_refs, n=50, n_hard=10, seed=42)
fx_raw = load_rows_by_offset(paths.train, ['ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz'],
                             sorted(int(q.split('_')[1]) for q in fx_ids)).rename(columns={'_row_offset': 'row_offset'})
fx_raw['query_id'] = 'train_' + fx_raw['row_offset'].astype(str)
fx = host_q[host_q['query_id'].isin(fx_ids)][['query_id', 'adduct', 'ionization_mode', 'instrument_type', 'collision_energy_ev', 'ingest_lib']].merge(
    fx_raw[['query_id', 'ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz']], on='query_id')
fx = fx.rename(columns={'query_id': 'spectrum_id', 'ingest_lib': 'exclude_source'}).assign(exclude_id=lambda d: d['spectrum_id'])
fx = fx.sort_values('spectrum_id').reset_index(drop=True)
bundle_obj = Bundle(BUNDLE, verify=True)
fx_man, fx_exp = build_fixture(bundle_obj, fx, BUNDLE)
print(f"fixture: {fx_man['n_queries']} queries, {fx_man['n_candidate_rows']:,} candidate rows; adducts {fx['adduct'].value_counts().to_dict()}")

# cross-check the frozen expectations against the TRAINING features (known T2 exceptions reported, never hidden)
qcr_t2 = pd.read_parquet(settlement['evidence_artifacts']['host_qcr'], columns=['query_id', 'candidate_key', 'tier', 'ref_source', 'query_source'])
t2 = set(map(tuple, qcr_t2[(qcr_t2['tier'] == 'T2') & (qcr_t2['ref_source'] != qcr_t2['query_source'])][['query_id', 'candidate_key']].drop_duplicates().to_numpy()))
chk = fx_exp.assign(candidate_connectivity_key=conn.set_index('conn_idx').loc[fx_exp['conn_idx'], 'connectivity_key'].to_numpy()).merge(
    host_feats, left_on=['spectrum_id', 'candidate_connectivity_key'], right_on=['query_id', 'candidate_connectivity_key'], suffixes=('_fx', '_train'))
bad = np.zeros(len(chk), bool)
for c in SPECTRAL_FEATURES:
    a, b = chk[f'{c}_fx'].to_numpy(float), chk[f'{c}_train'].to_numpy(float)
    bad |= ~((np.isnan(a) & np.isnan(b)) | (np.abs(a - b) <= 1e-9))
known = np.array([(q, k) in t2 for q, k in zip(chk['spectrum_id'], chk['candidate_connectivity_key'])], dtype=bool)
print(f'fixture vs training features: {len(chk):,} rows compared, {int((bad & ~known).sum())} unexplained mismatches, '
      f'{int((bad & known).sum())} known-T2 rows differ, {len(fx_exp) - len(chk)} fixture rows outside the frozen HOST pool')
assert int((bad & ~known).sum()) == 0

old = json.loads((BUNDLE / 'manifest.json').read_text(encoding='utf-8'))
manifest = bx.refresh_manifest(BUNDLE, extra={'model_id': old.get('model_id'), 'bundle_version': old.get('bundle_version'),
                                              'selftest': {k: fx_man[k] for k in ('n_queries', 'n_candidate_rows', 'tolerances')}})
_, _, problems = verify_bundle(BUNDLE)
print('bundle verification after fixture export:', problems or 'OK')
assert not problems

c:\Users\myben\OneDrive\Documents\Enveda\src\casmi_infer\selftest.py:251: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  strata = {k: list(g["query_id"].iloc[rng.permutation(len(g))]) for k, g in d.groupby(["adduct", "pool_q", "ref_bin", "ce_known"], sort=True)}


fixture: 50 queries, 8,396 candidate rows; adducts {'[M+H]+': 14, '[M-H]-': 6, '[M+NH4]+': 5, '[M-H2O+H]+': 5, '[M-2H2O+H]+': 4, '[M+K]+': 4, '[M+CH2O2-H]-': 4, '[M+Na]+': 4, '[M+Cl]-': 3, '[M-H2O-H]-': 1, '[M-2H]-': 0, '[M-C12H19O9]+': 0, '[M-C18H29O14]+': 0, '[2M+C2H4N]+': 0, '[M-C2H3O]-': 0, '[M-C3H6NO2]-': 0, '[M-C3H7O2]-': 0, '[M-C4H11O2]+': 0, '[M-C4H7]+': 0, '[M-C3H5]+': 0, '[M+Na-H2]-': 0, '[M+Na3]3+': 0, '[M+Na2]2+': 0, '[M+Na-H]+': 0, '[M-C5H9N2O]+': 0, '[M+Na-H2O]+': 0, '[M+N-O2]+': 0, '[M+Li]+': 0, '[M+Li-H]+': 0, '[M+K-H2]-': 0, '[M+HO]-': 0, '[M-C5H7O3]+': 0, '[M-C6H11O6]+': 0, '[M-C6H11O5]-': 0, '[M+HNa]2+': 0, '[M]+': 0, '[M-O]+': 0, '[M-O3P]+': 0, '[M-H9O5]+': 0, '[M-H7O4]+': 0, '[M-H5O3]+': 0, '[M-H4O3]2+': 0, '[M-H2]2-': 0, '[M-H2O]+': 0, '[M-H2O2]2+': 0, '[M-H2N]+': 0, '[M-H21]-': 0, '[M-CO2-H]-': 0, '[M-CH4]-': 0, '[M-CH3]-': 0, '[M-CH3O]+': 0, '[M-CH3F2O2]-': 0, '[M-C9H9O5]+': 0, '[M-C8H9O]+': 0, '[M-C7H5O]+': 0, '[M-C7H11O6]+': 0, '[M-C6H9O8S]+': 0, '[M-C6H9O5]+'